# Lesson 3.4 — Observability and Monitoring at Scale

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Design a structured log schema for LLM requests
- Select metrics and alerts that catch quality and cost regressions
- Handle observability challenges unique to LLMs at scale

**You will produce:** An `observability_schema.json` log record plus a metrics/alerting plan.

## Concepts

**Observability** is the ability to understand what your system did and why, from its outputs: logs, metrics, and traces. LLM systems add hard problems: outputs are non-deterministic, 'correctness' is fuzzy, costs scale with tokens, and prompts/models change under you.

A good **log schema** captures per request: ids, model + prompt version, token counts, latency, cost, retrieval sources, output, guardrail results, and (when available) an eval/human label. From these you derive metrics — p50/p95 latency, cost/request, refusal rate, guardrail-violation rate, retrieval-hit rate — and set alerts on **regressions** (e.g. sudden quality drop after a document refresh, the Sample 3 scenario). At scale you also sample traces, redact PII before logging, and watch for drift. Never log raw PHI/PII.

> **Reference:** [Anthropic — Monitoring & observability](https://docs.anthropic.com/en/docs/overview)

In [ ]:
# !pip install anthropic
import os, time, json

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Define the per-request log record

A structured record (not free-text logs) is what makes LLM systems analyzable. Note the prompt/model version fields — essential for correlating regressions to changes.

In [ ]:
import json, uuid, time
def log_record(user_hash, model, prompt_version, in_tok, out_tok, latency_ms,
               sources, guardrail_ok, eval_label=None):
    return {
      "request_id": uuid.uuid4().hex,
      "ts": time.time(),
      "user": user_hash,              # hashed, never raw PII/PHI
      "model": model,
      "prompt_version": prompt_version,
      "in_tokens": in_tok, "out_tokens": out_tok,
      "latency_ms": latency_ms,
      "retrieval_sources": sources,
      "guardrail_ok": guardrail_ok,
      "eval_label": eval_label,
    }

rec = log_record("u_8f3a", MODELS["sonnet"], "triage_v4", 1200, 180, 1550,
                 ["policy_7.pdf#p3"], True, eval_label="correct")
print(json.dumps(rec, indent=2))
with open("observability_schema.json","w",encoding="utf-8") as fh: json.dump(rec, fh, indent=2)

## Code-Along: Step 2 — Aggregate metrics and alert on regressions

From a batch of records we compute the metrics that matter and raise alerts on threshold breaches — the on-call signal for quality or cost problems at scale.

In [ ]:
import statistics
batch = [
  log_record("u1", MODELS['sonnet'], 'triage_v4', 1200, 150, 1400, ['a'], True, 'correct'),
  log_record("u2", MODELS['sonnet'], 'triage_v4', 1300, 160, 2100, ['b'], True, 'correct'),
  log_record("u3", MODELS['sonnet'], 'triage_v4', 1250, 170, 2600, [],    True, 'wrong'),
  log_record("u4", MODELS['sonnet'], 'triage_v4', 1280, 155, 1800, ['c'], False,'wrong'),
]
def metrics(batch):
    lat = sorted(r['latency_ms'] for r in batch)
    p95 = lat[int(0.95*(len(lat)-1))]
    acc = sum(r['eval_label']=='correct' for r in batch)/len(batch)
    guard = sum(not r['guardrail_ok'] for r in batch)/len(batch)
    nohit = sum(not r['retrieval_sources'] for r in batch)/len(batch)
    return {"p95_ms":p95, "accuracy":acc, "guardrail_violation":guard, "retrieval_miss":nohit}

m = metrics(batch); print(m)
alerts = []
if m['accuracy'] < 0.9: alerts.append('accuracy regression')
if m['p95_ms'] > 2500: alerts.append('latency SLA breach')
if m['retrieval_miss'] > 0.1: alerts.append('retrieval returning no sources (see Sample 3)')
print('ALERTS:', alerts)

## Exercise

Add **cost tracking**: compute cost per request from token counts (placeholder prices) and alert when rolling cost/request rises >25% versus a baseline. Also add a `prompt_version` breakdown so a regression can be attributed to a specific prompt change.

In [ ]:
# TODO: per-request cost from in/out tokens (verify prices at https://docs.anthropic.com)
# TODO: alert on cost/request rising >25% vs baseline
# TODO: group accuracy by prompt_version to localize a regression

## Key Takeaways

- Log structured records with model + prompt version to correlate regressions to changes.
- Derive LLM-specific metrics: p95 latency, cost/request, guardrail and retrieval-miss rates.
- Alert on regressions — a retrieval-miss spike is the Sample 3 failure signal.
- Redact PII/PHI before logging; sample traces to stay affordable at scale.

---

**Next:** [Lesson 3.5 — RAG Pipeline: Chunking, Indexing, and Retrieval](./lesson-3.5-rag-pipeline-retrieval.ipynb)